In [ ]:
import io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

# ==========================================
# 1. MODEL ARCHITECTURE (STUDENT)
# ==========================================
class FeatureFusionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.res_block = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = x + self.conv(skip)
        return x + self.res_block(x)

class MobileMiDaS(nn.Module):
    def __init__(self):
        super().__init__()
        # Load lightweight backbone for edge-device inference
        mnet = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).features

        self.enc1 = mnet[0:2]   # 1/2 res, 16 channels
        self.enc2 = mnet[2:4]   # 1/4 res, 24 channels
        self.enc3 = mnet[4:9]   # 1/8 res, 48 channels
        self.enc4 = mnet[9:]    # 1/16 res, 576 channels

        self.bottleneck = nn.Conv2d(576, 128, kernel_size=1)

        self.fusion3 = FeatureFusionBlock(48, 128)
        self.fusion2 = FeatureFusionBlock(24, 128)
        self.fusion1 = FeatureFusionBlock(16, 128)

        self.head = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU() # Depth must be positive
        )

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)

        out = self.bottleneck(s4)
        out = self.fusion3(out, s3)
        out = self.fusion2(out, s2)
        out = self.fusion1(out, s1)

        # Upsample back to input resolution
        out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        return self.head(out)

In [ ]:
# ==========================================
# 2. DATASET WRAPPER
# ==========================================
class InfinigenDistillationDataset(Dataset):
    def __init__(self, split="train", target_size=(256, 256)):
        self.ds = load_dataset("prs-eth/Pano-Infinigen", name="indoor", split=split)
        self.target_size = target_size

        # We will use Depth Anything V2 via HF Transformers as the Teacher
        self.teacher_processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]

        # 1. RGB Image
        image_pil = sample["image"].convert("RGB")
        image_resized = image_pil.resize(self.target_size)
        image_tensor = torch.from_numpy(np.array(image_resized)).permute(2, 0, 1).float() / 255.0

        # 2. Synthetic Ground Truth Depth (from Infinigen)
        depth_np = np.load(io.BytesIO(sample["depth"])).astype(np.float32)
        depth_tensor = torch.from_numpy(depth_np).unsqueeze(0)
        depth_tensor = F.interpolate(depth_tensor.unsqueeze(0), size=self.target_size, mode='nearest').squeeze(0)

        # 3. Prepare Teacher inputs
        teacher_inputs = self.teacher_processor(images=image_pil, return_tensors="pt")

        return {
            "image": image_tensor,
            "synthetic_depth": depth_tensor,
            "teacher_pixel_values": teacher_inputs["pixel_values"].squeeze(0)
        }

In [ ]:
# ==========================================
# 3. LOSS FUNCTIONS
# ==========================================
def compute_scale_and_shift_invariant_loss(pred, target):
    """Handles disparate scales between Teacher outputs and Synthetic Data."""
    mask = target > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)

    log_pred = torch.log(pred[mask] + 1e-6)
    log_gt = torch.log(target[mask] + 1e-6)

    diff = log_pred - log_gt
    loss = torch.sqrt((diff**2).mean() - 0.5 * (diff.mean()**2))
    return loss

def compute_gradient_loss(pred, target):
    """Forces the model to learn sharp boundaries (edges of doors, poles, etc.)."""
    pred_dx = torch.abs(pred[:, :, :, :-1] - pred[:, :, :, 1:])
    pred_dy = torch.abs(pred[:, :, :-1, :] - pred[:, :, 1:, :])

    target_dx = torch.abs(target[:, :, :, :-1] - target[:, :, :, 1:])
    target_dy = torch.abs(target[:, :, :-1, :] - target[:, :, 1:, :])

    loss_dx = torch.mean(torch.abs(pred_dx - target_dx))
    loss_dy = torch.mean(torch.abs(pred_dy - target_dy))
    return loss_dx + loss_dy

In [ ]:
# ==========================================
# 4. TRAINING LOOP
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    # Initialize Models
    student_model = MobileMiDaS().to(device)
    teacher_model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf").to(device)
    teacher_model.eval() # Teacher is frozen

    # Dataloader
    dataset = InfinigenDistillationDataset(split="train")
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)

    # Optimizer & Hyperparameters
    optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs = 10
    alpha = 0.5 # Weight between Teacher Pseudo-GT and Synthetic GT

    student_model.train()

    for epoch in range(epochs):
        epoch_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in progress_bar:
            images = batch["image"].to(device)
            synthetic_depth = batch["synthetic_depth"].to(device)
            teacher_inputs = batch["teacher_pixel_values"].to(device)

            # --- 1. Teacher Forward Pass (No Gradients) ---
            with torch.no_grad():
                teacher_outputs = teacher_model(pixel_values=teacher_inputs)
                pseudo_gt = teacher_outputs.predicted_depth
                # Resize pseudo-GT to match student output size
                pseudo_gt = F.interpolate(pseudo_gt.unsqueeze(1), size=(256, 256), mode='bilinear', align_corners=False)

            # --- 2. Student Forward Pass ---
            student_pred = student_model(images)

            # --- 3. Compute Mixed Loss ---
            # Loss vs Infinigen (Perfect Geometry)
            loss_synth_ssi = compute_scale_and_shift_invariant_loss(student_pred, synthetic_depth)
            loss_synth_grad = compute_gradient_loss(student_pred, synthetic_depth)
            loss_synth = loss_synth_ssi + (0.5 * loss_synth_grad)

            # Loss vs Teacher (Semantic Understanding)
            loss_teacher = compute_scale_and_shift_invariant_loss(student_pred, pseudo_gt)

            # Total Distillation Loss
            total_loss = (alpha * loss_synth) + ((1 - alpha) * loss_teacher)

            # --- 4. Backpropagation ---
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            epoch_loss += total_loss.item()
            progress_bar.set_postfix(loss=total_loss.item())

        print(f"Epoch {epoch+1} Average Loss: {epoch_loss / len(dataloader):.4f}")

        # Save checkpoint
        torch.save(student_model.state_dict(), f"mobile_midas_student_ep{epoch+1}.pth")

if __name__ == "__main__":
    train()

Training on: cuda
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 128MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

...

Epoch 8 Average Loss: 0.9478


Epoch 9/10: 100%|██████████| 6/6 [00:23<00:00,  3.95s/it, loss=0.964]


Epoch 9 Average Loss: 0.9403


Epoch 10/10: 100%|██████████| 6/6 [00:23<00:00,  3.93s/it, loss=0.93]


Epoch 10 Average Loss: 0.9309


In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset
import torch.nn.functional as F

class ReDWebDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(256, 256)):
        self.root_dir = root_dir
        self.target_size = target_size

        # Paths for RGB images and Relative Depth maps
        self.image_dir = os.path.join(root_dir, "Imgs")
        self.depth_dir = os.path.join(root_dir, "RDs")
        self.filenames = [f for f in os.listdir(self.image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        depth_name = img_name.replace('.jpg', '.png') # RD maps are usually .png

        # Load RGB
        image = cv2.imread(os.path.join(self.image_dir, img_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load Relative Depth (RD)
        depth = cv2.imread(os.path.join(self.depth_dir, depth_name), cv2.IMREAD_UNCHANGED)

        # Resize
        image = cv2.resize(image, self.target_size)
        depth = cv2.resize(depth, self.target_size, interpolation=cv2.INTER_NEAREST)

        # Normalize and convert to tensors
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        depth_tensor = torch.from_numpy(depth).unsqueeze(0).float()

        return {"image": image_tensor, "depth": depth_tensor}

In [ ]:
def train_stage_two(model, redweb_loader, optimizer, device):
    model.train()
    for batch in redweb_loader:
        images = batch["image"].to(device)
        targets = batch["depth"].to(device)

        # 1. ImageNet Normalization
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
        images = (images - mean) / std

        # 2. Forward Pass
        predictions = model(images)

        # 3. SSI Loss + Gradient Loss
        # (Using the functions from the previous training script)
        loss_ssi = compute_scale_and_shift_invariant_loss(predictions, targets)
        loss_grad = compute_gradient_loss(predictions, targets)

        total_loss = loss_ssi + (0.5 * loss_grad)

        # 4. Backprop
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

In [ ]:
import tarfile

def extract_dataset(file_path, extract_path="."):
    if file_path.endswith("tar.gz"):
        with tarfile.open(file_path, "r:gz") as tar:
            tar.extractall(path=extract_path)
            print(f"Extraction complete: {extract_path}")

# Usage
extract_dataset("ReDWeb_V1.tar.gz", "./data/ReDWeb_V1")

/tmp/ipykernel_219/2448452676.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extraction complete: ./data/ReDWeb_V1


In [ ]:
import torch
from torch.utils.data import DataLoader

# 1. Initialize the ReDWeb Dataset and DataLoader
# Ensure you have extracted your ReDWeb files to the 'root_dir'
redweb_train_ds = ReDWebDataset(root_dir="./data/ReDWeb_V1/ReDWeb_V1", split="train")
redweb_loader = DataLoader(redweb_train_ds, batch_size=4, shuffle=True, num_workers=4)

# 2. Setup Device and Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MobileMiDaS().to(device)

# 3. LOAD STAGE 1 WEIGHTS (Essential)
# This loads the geometric knowledge learned from the synthetic Infinigen dataset
checkpoint_path = "mobile_midas_student_ep10.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print(f"Successfully loaded Stage 1 weights from {checkpoint_path}")

# 4. Optimizer with a lower learning rate for fine-tuning
# We use a smaller LR (1e-5) to avoid destroying the metric features from Stage 1
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)

# 5. Execute the Training Function
# This calls the Stage 2 logic which uses Scale-Shift Invariant Loss
num_epochs_fine_tuning = 5

for epoch in range(num_epochs_fine_tuning):
    print(f"Starting Fine-Tuning Epoch {epoch+1}/{num_epochs_fine_tuning}")

    # Call the training function defined in your pipeline
    train_stage_two(
        model=model,
        redweb_loader=redweb_loader,
        optimizer=optimizer,
        device=device
    )

    # Save the updated hybrid model
    torch.save(model.state_dict(), f"mobile_midas_hybrid_ep{epoch+1}.pth")

Successfully loaded Stage 1 weights from mobile_midas_student_ep10.pth
Starting Fine-Tuning Epoch 1/5
Starting Fine-Tuning Epoch 2/5
Starting Fine-Tuning Epoch 3/5
Starting Fine-Tuning Epoch 4/5
Starting Fine-Tuning Epoch 5/5


In [ ]:
import io
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from datasets import load_dataset
from tqdm import tqdm

# ==========================================
# 1. ARCHITECTURE: MobileMiDaS with Sharp Decoder
# ==========================================

class SharpFusionBlock(nn.Module):
    """Fuses low-res features with high-res skip connections using PixelShuffle."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # PixelShuffle(2) doubles H,W resolution by rearranging channels
        self.upsample = nn.Sequential(
            nn.Conv2d(in_channels, out_channels * 4, kernel_size=1),
            nn.PixelShuffle(2)
        )
        self.conv_skip = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        # Ensure spatial dimensions match exactly
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = x + self.conv_skip(skip)
        return self.refine(x)

class MobileMiDaS_Sharp(nn.Module):
    def __init__(self):
        super().__init__()
        # Backbone: MobileNetV3-Small for RPi 4 efficiency
        mnet = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).features

        # Encoder Stages (Skip Connections)
        self.enc1 = mnet[0:2]   # 1/2 res, 16 ch
        self.enc2 = mnet[2:4]   # 1/4 res, 24 ch
        self.enc3 = mnet[4:9]   # 1/8 res, 48 ch
        self.enc4 = mnet[9:]    # 1/16 res, 576 ch

        self.bottleneck = nn.Conv2d(576, 128, kernel_size=1)

        # Decoder Stages (Symmetric to Encoder)
        self.fusion3 = SharpFusionBlock(128, 48) # 1/16 -> 1/8
        self.fusion2 = SharpFusionBlock(48, 24)  # 1/8 -> 1/4
        self.fusion1 = SharpFusionBlock(24, 16)  # 1/4 -> 1/2

        self.head = nn.Sequential(
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.ReLU() # Depth is positive
        )

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)

        out = self.bottleneck(s4)
        out = self.fusion3(out, s3)
        out = self.fusion2(out, s2)
        out = self.fusion1(out, s1)

        # Final 2x upsample to match input 256x256
        out = F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)
        return self.head(out)

# ==========================================
# 2. DATASET: Handling <f2 (float16) Binary
# ==========================================

class PerspectiveInfinigenDataset(Dataset):
    def __init__(self, split="train", size=(256, 256)):
        # Loading perspective (non-pano) split
        self.ds = load_dataset("infinigen/infinigen", split=split)
        self.size = size
        # ImageNet Norm constants
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]

        # 1. Process RGB
        img = sample["image"].convert("RGB").resize(self.size)
        img_np = np.array(img).transpose(2, 0, 1) / 255.0
        img_tensor = torch.from_numpy((img_np - self.mean) / self.std).float()

        # 2. Process Binary Depth (The fix for blur/corruption)
        # Infinigen stores depth as .npy binary float16
        depth_raw = np.load(io.BytesIO(sample["depth"])).astype(np.float32)
        depth_tensor = torch.from_numpy(depth_raw).unsqueeze(0).unsqueeze(0)

        # Resize GT using NEAREST to keep edges mathematically sharp
        depth_rescaled = F.interpolate(depth_tensor, size=self.size, mode='nearest').squeeze(0)

        # Normalization for Blind-Aid: Focus on 0-25m for nearby obstacles
        depth_capped = torch.clamp(depth_rescaled, 0.01, 25.0)
        # Using Log-scaling makes the model very sensitive to close-range objects
        depth_norm = torch.log1p(depth_capped) / np.log1p(25.0)

        return img_tensor, depth_norm

# ==========================================
# 3. TRAINING LOGIC: SSI + Gradient Loss
# ==========================================

def compute_losses(pred, target):
    # 1. Scale-Shift Invariant Loss (SSI)
    # Essential because different cameras/datasets have different internal scales
    mask = target > 0
    if not mask.any(): return torch.tensor(0.0).to(pred.device)

    log_p, log_t = torch.log(pred[mask] + 1e-6), torch.log(target[mask] + 1e-6)
    diff = log_p - log_t
    ssi_loss = torch.sqrt((diff**2).mean() - 0.5 * (diff.mean()**2))

    # 2. Gradient Loss (The Edge Sharpener)
    # Matches the 'jumps' in depth (edges) between prediction and ground truth
    p_dx = pred[:, :, :, :-1] - pred[:, :, :, 1:]
    p_dy = pred[:, :, :-1, :] - pred[:, :, 1:, :]
    t_dx = target[:, :, :, :-1] - target[:, :, :, 1:]
    t_dy = target[:, :, :-1, :] - target[:, :, 1:, :]

    grad_loss = torch.mean(torch.abs(p_dx - t_dx)) + torch.mean(torch.abs(p_dy - t_dy))

    return ssi_loss + 0.5 * grad_loss

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = MobileMiDaS_Sharp().to(device)
    dataset = PerspectiveInfinigenDataset()
    loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Training Loop
    epochs = 15
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for imgs, gts in pbar:
            imgs, gts = imgs.to(device), gts.to(device)

            preds = model(imgs)
            loss = compute_losses(preds, gts)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        # Save every epoch so you can validate on ReDWeb
        torch.save(model.state_dict(), f"mobile_midas_sharp_ep{epoch+1}.pth")
        print(f"Avg Loss: {epoch_loss/len(loader):.4f}")

if __name__ == "__main__":
    main()

Using device: cuda


DatasetNotFoundError: Dataset 'infinigen/infinigen' doesn't exist on the Hub or cannot be accessed.

#REDWEB

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from tqdm import tqdm
import numpy as np

# ==========================================
# 1. ARCHITECTURE: MobileMiDaS Sharp
# ==========================================
class SharpFusionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.Sequential(
            nn.Conv2d(in_channels, out_channels * 4, kernel_size=1),
            nn.PixelShuffle(2)
        )
        self.conv_skip = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.refine(x + self.conv_skip(skip))

class MobileMiDaS_ReDWeb(nn.Module):
    def __init__(self):
        super().__init__()
        mnet = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).features
        self.enc1, self.enc2, self.enc3, self.enc4 = mnet[0:2], mnet[2:4], mnet[4:9], mnet[9:]
        self.bottleneck = nn.Conv2d(576, 128, kernel_size=1)
        self.fusion3 = SharpFusionBlock(128, 48)
        self.fusion2 = SharpFusionBlock(48, 24)
        self.fusion1 = SharpFusionBlock(24, 16)
        self.head = nn.Sequential(
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid() # Relative depth mapped 0-1
        )

    def forward(self, x):
        s1, s2, s3, s4 = self.enc1(x), self.enc2(self.enc1(x)), self.enc3(self.enc2(self.enc1(x))), self.enc4(self.enc3(self.enc2(self.enc1(x))))
        # Trace logic for speed
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        out = self.bottleneck(s4)
        out = self.fusion3(out, s3)
        out = self.fusion2(out, s2)
        out = self.fusion1(out, s1)
        return self.head(F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False))

# ==========================================
# 2. DATASET: ReDWeb V1 Loader
# ==========================================
class ReDWebDataset(Dataset):
    def __init__(self, root_dir, size=(256, 256)):
        self.image_dir = os.path.join(root_dir, "Imgs")
        self.depth_dir = os.path.join(root_dir, "RDs")
        self.filenames = [f for f in os.listdir(self.image_dir) if f.endswith('.jpg')]
        self.size = size
        # ImageNet normalization
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        # ReDWeb depth maps are usually .png or .npy, check your extracted folder
        depth_name = img_name.replace('.jpg', '.png')

        img = cv2.imread(os.path.join(self.image_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.size)
        img_tensor = torch.from_numpy((img.transpose(2,0,1)/255.0 - self.mean)/self.std).float()

        depth = cv2.imread(os.path.join(self.depth_dir, depth_name), cv2.IMREAD_UNCHANGED)
        depth = cv2.resize(depth, self.size, interpolation=cv2.INTER_NEAREST)
        # Normalize relative depth to 0-1
        depth_tensor = torch.from_numpy(depth).float().unsqueeze(0)
        depth_tensor = (depth_tensor - depth_tensor.min()) / (depth_tensor.max() - depth_tensor.min() + 1e-8)

        return img_tensor, depth_tensor

# ==========================================
# 3. LOSS: Ranking/Ordinal Loss
# ==========================================
def compute_ranking_loss(prediction, target):
    # Ensure target matches prediction resolution (e.g., 256x256)
    if prediction.shape[-2:] != target.shape[-2:]:
        target = F.interpolate(target, size=prediction.shape[2:], mode='nearest')

    mask = target > 0

    # Flatten both to 1D based on the mask to avoid index errors
    # This effectively 'squeezes' out the invalid pixels
    p_valid = prediction[mask]
    t_valid = target[mask]

    if p_valid.numel() == 0:
        return torch.tensor(0.0, device=prediction.device, requires_grad=True)

    log_p = torch.log(p_valid + 1e-6)
    log_t = torch.log(t_valid + 1e-6)

    diff = log_p - log_t
    # Scale-Invariant Log Loss calculation
    loss = torch.sqrt((diff**2).mean() - 0.5 * (diff.mean()**2))
    return loss

# ==========================================
# 4. TRAINING EXECUTION
# ==========================================
def train_redweb(data_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MobileMiDaS_ReDWeb().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    dataset = ReDWebDataset(root_dir=data_path)
    loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4)

    for epoch in range(20):
        model.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/20")
        for imgs, gts in pbar:
            imgs, gts = imgs.to(device), gts.to(device)
            preds = model(imgs)
            loss = compute_ranking_loss(preds, gts)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=loss.item())

        torch.save(model.state_dict(), f"mobile_midas_redweb_ep{epoch+1}.pth")

if __name__ == "__main__":
    # Update this path to where you extracted the .tar.gz
    train_redweb("./data/ReDWeb_V1/ReDWeb_V1")

Epoch 20/20: 100%|██████████| 225/225 [00:28<00:00,  7.95it/s, loss=0.351]
